In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

model = joblib.load('models/lgbm_delay_classifier_final.pkl')

FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year',
    'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index',
]
CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']

flights = pd.read_parquet('dataset/merged_flights_fe_v2.parquet')
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

test = flights[flights['FL_DATE'] >= '2025-01-01']
X_test = test[FEATURES]
y_test = test['ARR_DEL15']
arr_delay = test['ARR_DELAY']

y_pred = model.predict_proba(X_test)[:, 1]

del flights
print(f"Test set: {len(y_test):,} flights")
print("✓ Ready for cost-benefit analysis")

Test set: 4,519,126 flights
✓ Ready for cost-benefit analysis


In [ ]:
COST_PER_MINUTE = 100.76
PROACTIVE_SAVINGS_RATE = 0.35
FALSE_ALARM_COST = 250 

THRESHOLD = 0.65
y_bin = (y_pred >= THRESHOLD).astype(int)

tn_mask = (y_bin == 0) & (y_test == 0)
fp_mask = (y_bin == 1) & (y_test == 0)
fn_mask = (y_bin == 0) & (y_test == 1)
tp_mask = (y_bin == 1) & (y_test == 1)

tp = tp_mask.sum()
fp = fp_mask.sum()
fn = fn_mask.sum()
tn = tn_mask.sum()

tp_delays = arr_delay[tp_mask].clip(lower=0)  
fn_delays = arr_delay[fn_mask].clip(lower=0)

all_delayed = arr_delay[y_test == 1].clip(lower=0)
total_delay_minutes_no_model = all_delayed.sum()
total_cost_no_model = total_delay_minutes_no_model * COST_PER_MINUTE

tp_delay_cost = (tp_delays.sum() * COST_PER_MINUTE)
tp_savings = tp_delay_cost * PROACTIVE_SAVINGS_RATE
fp_wasted = fp * FALSE_ALARM_COST
fn_full_cost = fn_delays.sum() * COST_PER_MINUTE

total_cost_with_model = (tp_delay_cost - tp_savings) + fp_wasted + fn_full_cost
net_savings = total_cost_no_model - total_cost_with_model

print(f"{'=' * 65}")
print(f"COST-BENEFIT ANALYSIS")
print(f"Source: Airlines for America (A4A), 2024 DOT Form 41 data")
print(f"Cost rate: ${COST_PER_MINUTE}/min | Savings rate: {PROACTIVE_SAVINGS_RATE:.0%} | FA cost: ${FALSE_ALARM_COST}")
print(f"{'=' * 65}")

print(f"\nSCENARIO 1 — NO MODEL (status quo):")
print(f"  Total delay minutes:     {total_delay_minutes_no_model:>15,.0f}")
print(f"  Total delay cost:        ${total_cost_no_model:>15,.0f}")

print(f"\nSCENARIO 2 — WITH MODEL (threshold {THRESHOLD}):")
print(f"  TP ({tp:,} flights):")
print(f"    Delay cost:            ${tp_delay_cost:>15,.0f}")
print(f"    Saved (35%):           -${tp_savings:>14,.0f}")
print(f"  FP ({fp:,} flights):")
print(f"    False alarm cost:      ${fp_wasted:>15,.0f}")
print(f"  FN ({fn:,} flights):")
print(f"    Missed delay cost:     ${fn_full_cost:>15,.0f}")

print(f"\n{'=' * 65}")
print(f"  TOTAL COST WITHOUT MODEL: ${total_cost_no_model:>13,.0f}")
print(f"  TOTAL COST WITH MODEL:    ${total_cost_with_model:>13,.0f}")
print(f"  NET SAVINGS:              ${net_savings:>13,.0f}")
print(f"  SAVINGS PERCENTAGE:       {net_savings/total_cost_no_model*100:>12.1f}%")
print(f"{'=' * 65}")

annual_savings = net_savings * (12/8)
print(f"\n  ANNUALIZED SAVINGS:       ${annual_savings:>13,.0f}")
print(f"  PER FLIGHT SAVED (TP):    ${tp_savings/tp:>13,.0f}")
print(f"  PER FALSE ALARM:          ${FALSE_ALARM_COST:>13,.0f}")

print(f"\n  For every $1 spent on false alarms (${fp_wasted:,.0f}):")
print(f"  the model returns ${tp_savings/fp_wasted:.1f} in delay savings")

COST-BENEFIT ANALYSIS
Source: Airlines for America (A4A), 2024 DOT Form 41 data
Cost rate: $100.76/min | Savings rate: 35% | FA cost: $250

SCENARIO 1 — NO MODEL (status quo):
  Total delay minutes:          76,931,708
  Total delay cost:        $  7,751,638,898

SCENARIO 2 — WITH MODEL (threshold 0.65):
  TP (638,975 flights):
    Delay cost:            $  5,472,906,156
    Saved (35%):           -$ 1,915,517,155
  FP (250,079 flights):
    False alarm cost:      $     62,519,750
  FN (398,861 flights):
    Missed delay cost:     $  2,278,732,742

  TOTAL COST WITHOUT MODEL: $7,751,638,898
  TOTAL COST WITH MODEL:    $5,898,641,493
  NET SAVINGS:              $1,852,997,405
  SAVINGS PERCENTAGE:               23.9%

  ANNUALIZED SAVINGS:       $2,779,496,107
  PER FLIGHT SAVED (TP):    $        2,998
  PER FALSE ALARM:          $          250

  For every $1 spent on false alarms ($62,519,750):
  the model returns $30.6 in delay savings
